In [ ]:

import Module_dynamics as dyn
import os
import pandas as pd
import numpy as np
from glob import glob

import plotly.graph_objs as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative

# -------------------------------------------------------------------
# Paramètres
# -------------------------------------------------------------------
SIMNAME = "mass_100_motor_250_solar_0.5_battery_500.0_PROTOUR"
FOLDER = "SIMRESULTS"

# fenêtre temporelle (en secondes)
t_min = 0.0
t_max = 60 * 60 * 24 * 3

# -------------------------------------------------------------------
# Chargement des simulations compagnons
# -------------------------------------------------------------------
sims = glob(os.path.join(FOLDER, f"{SIMNAME}*"))
sims_dict = {}
for sim in sims:
    suffix = sim.split('_')[-1].split('.')[0]
    data = pd.read_csv(sim)
    sims_dict[suffix] = data

if not sims_dict:
    raise RuntimeError(f"Aucun fichier trouvé pour {SIMNAME} dans {FOLDER}")

# palette de couleurs cohérente entre les figures
base_colors = qualitative.Plotly  # ou qualitative.D3, etc.
colors = {suffix: base_colors[i % len(base_colors)]
          for i, suffix in enumerate(sims_dict.keys())}

# -------------------------------------------------------------------
# Figure 1 : variables vs temps
# -------------------------------------------------------------------
fig_t = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=(
        "Position [m]",
        "Vitesse [m/s]",
        "Accélération [m/s²]",
        "SOC [- ou %]",
        "Puissance moteur [W]"
    )
)

for idx, (suffix, data) in enumerate(sims_dict.items(), start=1):
    # conversions de base
    time = pd.to_numeric(data['simtime'], errors='coerce').to_numpy()
    position = pd.to_numeric(data['Distance'], errors='coerce').to_numpy()
    speed = pd.to_numeric(data['Velocity'], errors='coerce').to_numpy()

    # colonnes optionnelles : SOC, Motorpower
    if 'SOC' in data.columns:
        soc = pd.to_numeric(data['SOC'], errors='coerce').to_numpy()
    else:
        soc = np.full_like(time, np.nan, dtype=float)

    if 'Motorpower' in data.columns:
        motor_power = pd.to_numeric(data['Motorpower'], errors='coerce').to_numpy()
    else:
        motor_power = np.full_like(time, np.nan, dtype=float)

    # masque temporel
    mask = (time >= t_min) & (time <= t_max)
    time_plot = time[mask]
    time_h = time_plot / 3600.0  # en heures

    position_plot = position[mask]
    speed_plot = speed[mask]
    soc_plot = soc[mask]
    motor_power_plot = motor_power[mask]

    # accélération (m/s²)
    if len(time_plot) > 2:
        acc_plot = np.gradient(speed_plot, time_plot, edge_order=2)
    else:
        acc_plot = np.full_like(speed_plot, np.nan)

    label = f"approche {suffix}"
    color = colors[suffix]

    # Position vs temps
    fig_t.add_trace(
        go.Scatter(
            x=time_h, y=position_plot,
            mode="lines",
            name=label,
            legendgroup=suffix,
            line=dict(color=color)
        ),
        row=1, col=1
    )

    # Vitesse vs temps
    fig_t.add_trace(
        go.Scatter(
            x=time_h, y=speed_plot,
            mode="lines",
            name=label,
            legendgroup=suffix,
            showlegend=False,  # déjà montré sur la première ligne
            line=dict(color=color)
        ),
        row=2, col=1
    )

    # Accélération vs temps
    fig_t.add_trace(
        go.Scatter(
            x=time_h, y=acc_plot,
            mode="lines",
            name=label,
            legendgroup=suffix,
            showlegend=False,
            line=dict(color=color)
        ),
        row=3, col=1
    )

    # SOC vs temps (si pas tout NaN)
    if not np.all(np.isnan(soc_plot)):
        fig_t.add_trace(
            go.Scatter(
                x=time_h, y=soc_plot,
                mode="lines",
                name=label,
                legendgroup=suffix,
                showlegend=False,
                line=dict(color=color)
            ),
            row=4, col=1
        )

    # Puissance moteur vs temps (si pas tout NaN)
    if not np.all(np.isnan(motor_power_plot)):
        fig_t.add_trace(
            go.Scatter(
                x=time_h, y=motor_power_plot,
                mode="lines",
                name=label,
                legendgroup=suffix,
                showlegend=False,
                line=dict(color=color)
            ),
            row=5, col=1
        )

fig_t.update_xaxes(title_text="Temps [h]", row=5, col=1)
fig_t.update_layout(
    height=900,
    width=1200,
    title_text="Comparaison des approches – variables vs temps",
    legend_title_text="Approches",
)

fig_t.show()

# -------------------------------------------------------------------
# Figure 2 : variables vs distance
# -------------------------------------------------------------------
fig_d = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=(
        "Vitesse [m/s]",
        "Accélération [m/s²]",
        "SOC [- ou %]",
        "Puissance moteur [W]"
    )
)

for suffix, data in sims_dict.items():
    time = pd.to_numeric(data['simtime'], errors='coerce').to_numpy()
    position = pd.to_numeric(data['Distance'], errors='coerce').to_numpy()
    speed = pd.to_numeric(data['Velocity'], errors='coerce').to_numpy()

    if 'SOC' in data.columns:
        soc = pd.to_numeric(data['SOC'], errors='coerce').to_numpy()
    else:
        soc = np.full_like(time, np.nan, dtype=float)

    if 'Motorpower' in data.columns:
        motor_power = pd.to_numeric(data['Motorpower'], errors='coerce').to_numpy()
    else:
        motor_power = np.full_like(time, np.nan, dtype=float)

    # même fenêtre temporelle pour la cohérence
    mask = (time >= t_min) & (time <= t_max)
    time_plot = time[mask]
    position_plot = position[mask]
    speed_plot = speed[mask]
    soc_plot = soc[mask]
    motor_power_plot = motor_power[mask]

    # tri par distance croissante
    sort_idx = np.argsort(position_plot)
    position_plot = position_plot[sort_idx]
    speed_plot = speed_plot[sort_idx]
    soc_plot = soc_plot[sort_idx]
    motor_power_plot = motor_power_plot[sort_idx]

    # accel vs distance : dérivée sur temps, puis triée
    if len(time_plot) > 2:
        acc_plot_raw = np.gradient(speed[mask], time_plot, edge_order=2)
        acc_plot = acc_plot_raw[sort_idx]
    else:
        acc_plot = np.full_like(speed_plot, np.nan)

    label = f"approche {suffix}"
    color = colors[suffix]

    # Vitesse vs distance
    fig_d.add_trace(
        go.Scatter(
            x=position_plot, y=speed_plot,
            mode="lines",
            name=label,
            legendgroup=suffix,
            line=dict(color=color)
        ),
        row=1, col=1
    )

    # Accélération vs distance
    fig_d.add_trace(
        go.Scatter(
            x=position_plot, y=acc_plot,
            mode="lines",
            name=label,
            legendgroup=suffix,
            showlegend=False,
            line=dict(color=color)
        ),
        row=2, col=1
    )

    # SOC vs distance
    if not np.all(np.isnan(soc_plot)):
        fig_d.add_trace(
            go.Scatter(
                x=position_plot, y=soc_plot,
                mode="lines",
                name=label,
                legendgroup=suffix,
                showlegend=False,
                line=dict(color=color)
            ),
            row=3, col=1
        )

    # Puissance moteur vs distance
    if not np.all(np.isnan(motor_power_plot)):
        fig_d.add_trace(
            go.Scatter(
                x=position_plot, y=motor_power_plot,
                mode="lines",
                name=label,
                legendgroup=suffix,
                showlegend=False,
                line=dict(color=color)
            ),
            row=4, col=1
        )

fig_d.update_xaxes(title_text="Distance [m]", row=4, col=1)
fig_d.update_layout(
    height=800,
    width=1200,
    title_text="Comparaison des approches – variables vs distance",
    legend_title_text="Approches",
)

fig_d.show()
